# Projet 3 — Détection de messages frauduleux
Notebook principal (Phase 1 et 2).
Exécutable de haut en bas sur Colab frais : toutes les données ont un repli.

**Règle d'or** : le test est gelé en cellule 6, jamais utilisé pour régler.

## Cellule 1 — Installation (Colab)
Sur Colab frais, décommenter puis exécuter. En local après `pip install -r requirements.txt` et `pip install -e .`, inutile.

In [3]:
!pip -q install "scikit-learn>=1.4" "jupytext>=1.16" 2>/dev/null || True
!pip -q install -e . 2>/dev/null || True
print("Installation OK (décommenter au premier run Colab).")

Installation OK (décommenter au premier run Colab).


## Cellule 2 — Imports et configuration

In [4]:
import sys
from pathlib import Path

# Rendre `src` importable même sans `pip install -e .`
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from src.data_io import load_all_train, get_frozen_test, get_evasion
from src.features import extract_indices
from src.rules import rule_score

SEED = 42
np.random.seed(SEED)

COST_FN = 20.0
COST_FP = 1.0

print(f"ROOT = {ROOT}")
print(f"Seed = {SEED} | COST_FN = {COST_FN} | COST_FP = {COST_FP}")

ROOT = /Users/zhuanzmima0000/Desktop/Projet_3_NLP_Devoir
Seed = 42 | COST_FN = 20.0 | COST_FP = 1.0


## Cellule 3 — Chargement des données
Trois jeux : train (UCI + local), test gelé, test d'évasion.
Le premier run génère `data/test_frozen.jsonl` et affiche son hash.

In [5]:
train_df = load_all_train()
print(f"\nTrain : {len(train_df)} messages "
      f"({int(train_df.label.sum())} fraudes, {(train_df.label==0).sum()} légitimes)")

test_df = get_frozen_test()
evasion_df = get_evasion()
print(f"Test : {len(test_df)} | Évasion : {len(evasion_df)}")

[UCI] 5574 messages chargés (747 spam).
[Local] génération depuis gabarits.

Train : 6354 messages (1007 fraudes, 5347 légitimes)
[Test gelé] 150 messages — hash cb97cbd5dc5ea50c…
Test : 150 | Évasion : 30


## Cellule 4 — Prétraitement : démonstration sur un exemple

In [6]:
example = ("URGENT ! Votre compte Wave sera suspendu. "
           "Cliquez http://bit.ly/xx et envoyez 5 000 FCFA.")
print("Message :", example)
print("\nIndices extraits :")
for k, v in extract_indices(example).items():
    print(f"  {k:20s} = {v}")
print("\nBaseline règles B1 :")
score, reasons = rule_score(example)
print(f"  score = {score:.2f}")
print(f"  raisons = {reasons}")

Message : URGENT ! Votre compte Wave sera suspendu. Cliquez http://bit.ly/xx et envoyez 5 000 FCFA.

Indices extraits :
  has_url              = 1
  has_money            = 1
  has_code_request     = 0
  has_urgency          = 1
  has_send_request     = 1
  has_congrats         = 0
  upper_ratio          = 0.14606741573033707
  digit_ratio          = 0.0449438202247191
  n_exclam             = 1
  n_links              = 1
  len                  = 89

Baseline règles B1 :
  score = 0.70
  raisons = ['lien', 'montant', 'urgence', 'demande_envoi']


## Cellule 5 — Petite EDA

In [7]:
print("Distribution des labels (train) :")
print(train_df.label.value_counts(normalize=True).rename("proportion"))
print("\nTest gelé :")
print(test_df.label.value_counts())
print("\nTest gelé — répartition par type :")
print(test_df.type.value_counts())
print("\nÉvasion :")
print(evasion_df.label.value_counts())

Distribution des labels (train) :
label
0    0.841517
1    0.158483
Name: proportion, dtype: float64

Test gelé :
label
0    100
1     50
Name: count, dtype: int64

Test gelé — répartition par type :
type
conversation      18
otp               14
notif_paiement    14
rappel_facture    13
livraison         13
faux_gain         12
marketing         12
trap              10
hand              10
phishing           9
notification       6
faux_agent         6
faux_emploi        5
faux_service       4
faux_proche        4
Name: count, dtype: int64

Évasion :
label
1    15
0    15
Name: count, dtype: int64


## Cellule 6 — Split train/val + vérification du hash du test
⚠️ **À faire une seule fois à la fin de la semaine 1** : noter le hash affiché
dans `EXPECTED_TEST_HASH` (`src/data_io.py`), committer, et ne plus jamais
toucher `data/test_frozen.jsonl`.

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"], train_df["label"],
    test_size=0.2, random_state=SEED, stratify=train_df["label"],
)
X_test, y_test = test_df["text"], test_df["label"]
print(f"Train = {len(X_train)} | Val = {len(X_val)} | Test = {len(X_test)}")

from src.data_io import file_sha256, DATA
print(f"\n🔒 Hash du test gelé : {file_sha256(DATA / 'test_frozen.jsonl')}")
print("   Recopier dans EXPECTED_TEST_HASH (src/data_io.py) puis commit.")

Train = 5083 | Val = 1271 | Test = 150

🔒 Hash du test gelé : cb97cbd5dc5ea50cda5cd7f477eb71d3358bb2337447e57c7361c8f366aef77f
   Recopier dans EXPECTED_TEST_HASH (src/data_io.py) puis commit.


## Cellule 7 — Modèles (Phase 2)
Baselines B2, B3, et approche moderne M1.

In [9]:
import time
from src.models import build_b2, build_b3, build_m1, predict_scores
from src.cost import choose_threshold, sensibilite_ratio, expected_cost
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

# -- Baseline B2 --
b2 = build_b2()
b2.fit(X_train, y_train)
b2_val_scores = predict_scores("B2", b2, X_val)
b2_val_pred = (b2_val_scores >= 0.5).astype(int)
print("=== B2 (TF-IDF mots + LR) ===")
print(f"P={precision_score(y_val, b2_val_pred):.3f} | R={recall_score(y_val, b2_val_pred):.3f} | F1={f1_score(y_val, b2_val_pred, average='macro'):.3f} | PR-AUC={average_precision_score(y_val, b2_val_scores):.3f}")

# -- Baseline B3 --
b3 = build_b3()
b3.fit(X_train, y_train)
b3_val_scores = predict_scores("B3", b3, X_val)
b3_val_pred = (b3_val_scores >= 0.5).astype(int)
print("\n=== B3 (TF-IDF car + LR) ===")
print(f"P={precision_score(y_val, b3_val_pred):.3f} | R={recall_score(y_val, b3_val_pred):.3f} | F1={f1_score(y_val, b3_val_pred, average='macro'):.3f} | PR-AUC={average_precision_score(y_val, b3_val_scores):.3f}")

# -- Modèle moderne M1 --
print("\nChargement et entraînement de M1 (peut prendre ~30s)...")
t0 = time.time()
m1_embedder, m1_clf = build_m1(X_train, y_train)
m1_val_scores = predict_scores("M1", (m1_embedder, m1_clf), X_val)
m1_val_pred = (m1_val_scores >= 0.5).astype(int)
print(f"\n=== M1 (Embeddings + LR) | Temps: {time.time()-t0:.1f}s ===")
print(f"P={precision_score(y_val, m1_val_pred):.3f} | R={recall_score(y_val, m1_val_pred):.3f} | F1={f1_score(y_val, m1_val_pred, average='macro'):.3f} | PR-AUC={average_precision_score(y_val, m1_val_scores):.3f}")

=== B2 (TF-IDF mots + LR) ===
P=0.960 | R=0.960 | F1=0.976 | PR-AUC=0.984

=== B3 (TF-IDF car + LR) ===
P=0.995 | R=0.970 | F1=0.990 | PR-AUC=0.992

Chargement et entraînement de M1 (peut prendre ~30s)...


/opt/anaconda3/envs/projet3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement du modèle SentenceTransformer...
Encodage des textes d'entraînement...
Entraînement de la Régression Logistique...

=== M1 (Embeddings + LR) | Temps: 403.2s ===
P=0.920 | R=0.970 | F1=0.967 | PR-AUC=0.987


## Cellule 8 — Seuil par coût et Évaluation finale
Choix du seuil sur la validation, puis application sur le test gelé.

In [10]:
# -- Seuil par coût --
print("\n--- Choix du seuil (Modèle M1) ---")
best_t = choose_threshold(y_val, m1_val_scores)
print(f"Seuil retenu (minimise le coût FN/FP=20) : {best_t:.2f}\n")
sensibilite_ratio(y_val, m1_val_scores)

# -- Évaluation sur le TEST GELÉ --
print("\n=== ÉVALUATION SUR LE TEST GELÉ (M1) ===")
m1_test_scores = predict_scores("M1", (m1_embedder, m1_clf), X_test)
m1_test_pred = (m1_test_scores >= best_t).astype(int)

cm = confusion_matrix(y_test, m1_test_pred)
tn, fp, fn, tp = cm.ravel()
print(f"Matrice de confusion :\n{cm}")
print(f"P fraude = {precision_score(y_test, m1_test_pred):.3f}")
print(f"R fraude = {recall_score(y_test, m1_test_pred):.3f}")
print(f"Coût total = {20*fn + 1*fp:.1f} (FN={fn}, FP={fp})")

# -- Analyse des pièges --
trap_mask = (test_df["type"] == "trap").values
traps_mal_classees = ((m1_test_pred == 1) & trap_mask).sum()
print(f"\nPièges (lien+montant, légitimes) : {traps_mal_classees} / {trap_mask.sum()} classés fraude par erreur.")


--- Choix du seuil (Modèle M1) ---
Seuil retenu (minimise le coût FN/FP=20) : 0.26

Sensibilité au ratio de coût (rappel fraude au seuil optimal) :
  FN/FP=  5 → seuil=0.47 | R=0.975 | P=0.916
  FN/FP= 10 → seuil=0.47 | R=0.975 | P=0.916
  FN/FP= 20 → seuil=0.26 | R=0.985 | P=0.835
  FN/FP= 50 → seuil=0.26 | R=0.985 | P=0.835

=== ÉVALUATION SUR LE TEST GELÉ (M1) ===
Matrice de confusion :
[[84 16]
 [ 2 48]]
P fraude = 0.750
R fraude = 0.960
Coût total = 56.0 (FN=2, FP=16)

Pièges (lien+montant, légitimes) : 8 / 10 classés fraude par erreur.


## Cellule 9 — Évasion, erreurs, démo, LLM (Phase 3)
Test d'évasion, analyse d'erreurs, app Gradio, alerte LLM.
En attendant, stub.

## Cellule 9 — Test d'évasion et Sauvegarde du modèle

In [11]:
import joblib
from src.data_io import DATA

# -- Sauvegarde du modèle B3 pour l'application Gradio --
model_dir = DATA / "processed"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(b3, model_dir / "model_b3.joblib")
print("✅ Modèle B3 sauvegardé pour l'application Gradio.")

# -- Test d'évasion --
print("\n=== TEST D'ÉVASION (30 messages obfusqués) ===")
X_ev = evasion_df["text"].tolist()
y_ev = evasion_df["label"].values

b2_ev_scores = predict_scores("B2", b2, X_ev)
b3_ev_scores = predict_scores("B3", b3, X_ev)
m1_ev_scores = predict_scores("M1", (m1_embedder, m1_clf), X_ev)

# On applique le seuil de 0.5 pour comparer
for name, scores in [("B2", b2_ev_scores), ("B3", b3_ev_scores), ("M1", m1_ev_scores)]:
    pred = (scores >= 0.5).astype(int)
    r_evasion = recall_score(y_ev, pred, zero_division=0)
    print(f"Rappel {name} sur messages obfusqués : {r_evasion:.3f}")

✅ Modèle B3 sauvegardé pour l'application Gradio.

=== TEST D'ÉVASION (30 messages obfusqués) ===
Rappel B2 sur messages obfusqués : 1.000
Rappel B3 sur messages obfusqués : 1.000
Rappel M1 sur messages obfusqués : 0.933


## Cellule 10 — Analyse d'erreurs (Top 10)
Le sujet exige au moins 10 erreurs examinées et classées.

In [12]:
print("\n=== ANALYSE D'ERREURS SUR LE TEST GELÉ ===")
df_errors = test_df.copy()
df_errors["score"] = m1_test_scores
df_errors["pred"] = m1_test_pred

# Filtrer uniquement les erreurs
errors = df_errors[df_errors.label != df_errors.pred].copy()

def classify_cause(row):
    txt = row["text"]
    if row["label"] == 0 and row["pred"] == 1 and row["type"] == "trap":
        return "Piège (lien+montant légitime)"
    if row["label"] == 1 and ("@" in txt or "\u200b" in txt or re.search(r"\d\s\d", txt)):
        return "Obfuscation"
    if len(txt) < 30:
        return "Message trop court"
    return "Ambiguïté sémantique"

import re
errors["cause"] = errors.apply(classify_cause, axis=1)

# Affichage des 10 erreurs
sample = errors.head(10)
for idx, row in sample.iterrows():
    print(f"[{row['cause']}] Vrai: {row['label']} | Prédit: {row['pred']} (score {row['score']:.2f})")
    print(f"  Texte: {row['text'][:100]}...\n")


=== ANALYSE D'ERREURS SUR LE TEST GELÉ ===
[Piège (lien+montant légitime)] Vrai: 0 | Prédit: 1 (score 0.68)
  Texte: Commande #MP240315-8855 expédiée. Total : 15 000 FCFA. Suivi : https://wave.com/recu/8841...

[Ambiguïté sémantique] Vrai: 0 | Prédit: 1 (score 0.31)
  Texte: Code de validation : 8024. Valable 5 minutes. Ne le communiquez à personne....

[Ambiguïté sémantique] Vrai: 0 | Prédit: 1 (score 0.51)
  Texte: Rappel : votre facture Wave de 1 500 000 FCFA est due le le 02/04....

[Ambiguïté sémantique] Vrai: 0 | Prédit: 1 (score 0.35)
  Texte: Colis #MP240309-8849 livré aujourd'hui. Merci de votre confiance....

[Piège (lien+montant légitime)] Vrai: 0 | Prédit: 1 (score 0.44)
  Texte: Reçu #MP240318-8858 : 100 000 FCFA reçus de Fatou Ndiaye. Vérifier : https://orange.sn/facture...

[Ambiguïté sémantique] Vrai: 0 | Prédit: 1 (score 0.51)
  Texte: Rappel : votre facture Expresso de 100 000 FCFA est due le 20/03 à 18:30....

[Ambiguïté sémantique] Vrai: 0 | Prédit: 1 (score 0.49)


## Cellule 11 — (Phase 4, rapport)
Bilan critique.

In [13]:
print("[Phase 4] Bilan critique à rédiger dans report/rapport.md.")

[Phase 4] Bilan critique à rédiger dans report/rapport.md.
